# AIFM Private Debt — Risk Notebook

Risk limits are defined in the fund's offering document and monitored against
internal thresholds. No regulatory VaR limit applies (unlike UCITS).

Key regulatory obligations under AIFMD:
- **Leverage**: gross and commitment method (Annex IV)
- **Stress testing**: market, liquidity, and counterparty scenarios (Annex VI)
- **Liquidity risk**: portfolio liquidity profile and redemption stress
- **Annex IV reporting**: quarterly to CSSF. AIFMD II (Directive 2024/927/EU)
  expanded requirements, adding granular data on liquidity management tools,
  loan origination, and delegation arrangements.

Regulatory framework:
- AIFMD: Directive 2011/61/EU
- AIFMD II: Directive 2024/927/EU
- Delegated Regulation: EU 231/2013
- Annex IV reporting: EU 231/2013, ESMA technical guidance v1.7 (July 2024)
- Annex VI stress testing: ESMA/2020/1498
- Luxembourg implementation: Law of 12 July 2013 on AIFMs (AIFM Law)

Dual UCITS/AIFM ManCo:
- CSSF Regulation 10-04 (organisational and prudential requirements)
- CSSF Regulation 22-05 (sustainability requirements, amending 10-04)

#### In this notebook

AIFM Private Debt Fund. Strategy: senior secured loans, HY bonds, CLOs.

In [ ]:
from src.config import VALUATION_DATE, QUARTER
from src.risk.reg_constants import CONFIDENCE, HORIZON, NOTICE, GROSS_LIMIT

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.ui.plot_style import (
    C,  FUND_COLORS, ACCENT, ACCENT2, ACCENT3, WARNING,
    apply_ax_style, section_title, fig_header, fig_footer,
    callout_box, threshold_vline, threshold_hline, breach_fill,
    pct_color, rag_color, util_color, liq_color, table_cell, sup_title,
)
from src.ui.nb_utils import save_fig, save_table, styled_table, save_table_png

import warnings
warnings.filterwarnings('ignore')

from src.setup_db import run as setup_db
setup_db()

from src.data.database import get_engine, query_positions, query_nav_history
from src.data.enrichment import get_risk_ready_df
from src.data.mock_bloomberg import MockBloomberg as Bloomberg
from src.risk.leverage_config import INSTRUMENT_SOURCE
from src.risk.risk_utils import (
    var_historical, var_parametric, var_scale,
    es_historical, es_scale,
    exception_report, full_backtest_report,
    stress_equity, stress_rates, stress_credit,
    stress_fx, stress_combined, stress_historical,
    days_to_liquidate, liquidity_buckets, redemption_stress, investor_concentration,
    liquidity_adjusted_var,
)
import src.risk.risk_utils as rk
import src.print_html_utils as phtml
import src.print_utils as prt
import src.risk.esg_utils as esg_u 
from src.risk.esg_utils import ESG_FIELDS, ESG_THRESHOLD_LOW
from src.ui.nb_utils import save_fig, save_table, styled_table

FUND_ID    = 'AIFM_PrivateDebt'
VALUATION_DATE      = '2026-05-13'
ENGINE     = get_engine()
BBG        = Bloomberg()

HORIZON    = 20

---

## 1. Load and Validate Positions

Positions are queried from SQLite, which is loaded daily from the fund administrator
Excel export. The flow is:

Fund admin Excel → load_positions() → SQLite → query_positions() → notebook

`get_risk_ready_df` runs the enrichment pipeline on the raw positions:
- liquid instruments (equities, bonds, ETFs): sensitivities fetched from Bloomberg
  (beta, modified duration, convexity, spread duration)
- illiquid instruments (loans, direct properties): fund admin data already embedded
  in the position file (rating, maturity, LTV, rental yield) used directly,
  no Bloomberg ticker available or needed

The output is a single enriched DataFrame per fund per date, ready for VaR,
stress testing, and liquidity analysis.

In [ ]:
positions = query_positions(ENGINE, FUND_ID, VALUATION_DATE)
risk_df   = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)
NAV       = risk_df['market_value_eur'].sum()

print(f"Fund           : {FUND_ID}")
print(f"Valuation date : {VALUATION_DATE}")
print(f"Positions      : {len(positions)}")
print(f"NAV (EUR)      : {NAV:,.0f}")
print(f"Asset classes  : {sorted(positions['asset_class'].unique())}")
print(f"Long exposure  : {risk_df[risk_df['market_value_eur'] > 0]['market_value_eur'].sum():,.0f}")
print(f"Short exposure : {risk_df[risk_df['market_value_eur'] < 0]['market_value_eur'].sum():,.0f}")

In [ ]:
# Asset class breakdown
breakdown = risk_df.groupby('asset_class').agg(
    market_value_eur=('market_value_eur', 'sum'),
    n_positions=('isin', 'count'),
).sort_values('market_value_eur', ascending=False)

breakdown['weight_pct'] = breakdown['market_value_eur'] / NAV * 100

print(f"{'Asset Class':<20} {'MV (EUR)':>15} {'Weight':>8} {'# Pos':>6}")
print('-' * 52)
for ac, row in breakdown.iterrows():
    print(f"{ac:<20} {row['market_value_eur']:>15,.0f} {row['weight_pct']:>7.1f}% {row['n_positions']:>6}")
print('-' * 52)
print(f"{'NAV':<20} {NAV:>15,.0f} {'100.0%':>8}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2))
colors = [ACCENT2 if v < 0 else ACCENT for v in breakdown['weight_pct']]
bars = ax.barh(breakdown.index, breakdown['weight_pct'],
               color=colors, height=0.45, alpha=0.85)
ax.axvline(0, color=C['dim'], lw=0.8)
ax.set_xlabel('Weight (% NAV)', fontsize=9)
ax.set_xlim(0, breakdown['weight_pct'].max() * 1.1)
section_title(ax, 'Asset Class Breakdown', fontsize=10)
ax.grid(False)
for bar, val in zip(bars, breakdown['weight_pct']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=8)
ax.grid(False)
plt.tight_layout()
save_fig(fig, FUND_ID, "01. Asset Breakdown PD", dpi=150)
plt.show()


---

## 2. VaR and Expected Shortfall

VaR and ES are computed using historical simulation and a multi-factor Monte Carlo
approximation on daily NAV returns. Under AIFMD there is no regulatory VaR limit,
but VaR is monitored against internal limits defined in the RMP and reported in Annex IV.

- Confidence: 99%
- Horizon: 1-day and 20-day (square root of time scaling)
- Method: historical simulation, 250-day rolling window

> **Monte Carlo VaR**: not computed in this notebook. The objective here is to
> demonstrate the risk manager workflow and regulatory monitoring requirements,
> not to build a full valuation engine. For a ManCo, these figures would come
> from a third-party risk system. For an investment manager or risk system
> provider, the model would be built or licensed internally.

> **Private debt note**: loans and CLOs have no daily market prices. NAV returns
> reflect mark-to-model valuations from the fund administrator, which smooth
> volatility and underestimate tail risk. VaR figures for this fund should be
> treated as indicative. Credit stress testing is the primary risk tool for
> private debt portfolios.

In [ ]:
nav_history = query_nav_history(ENGINE, FUND_ID)
nav_history['date'] = pd.to_datetime(nav_history['date'])
returns = nav_history['pnl_pct'].dropna().values

var_1d  = var_historical(returns[-250:], confidence=CONFIDENCE)
var_20d = var_scale(var_1d, horizon=HORIZON)
es_1d   = es_historical(returns[-250:], confidence=CONFIDENCE)
es_20d  = es_scale(es_1d, horizon=HORIZON)

print(f"{'Metric':<25} {'1d':>10} {'20d':>10}")
print(f"{'':25} {'(% NAV)':>10} {'(% NAV)':>10}")
print('-' * 46)
print(f"{'VaR Historical':<25} {var_1d*100:>9.2f}% {var_20d*100:>9.2f}%")
print(f"{'ES Historical':<25} {es_1d*100:>9.2f}% {es_20d*100:>9.2f}%")
print('-' * 46)
print(f"{'VaR Hist (EUR)':<25} {var_1d*NAV:>10,.0f} {var_20d*NAV:>10,.0f}")


### 2.2 Liquidity-Adjusted VaR

Standard VaR assumes positions can be liquidated instantly at market price.
LVaR extends this by adding the estimated cost of unwinding positions under
stressed market conditions:

$$\text{LVaR} = \text{VaR} + \text{Liquidity Cost}$$

$$\text{Liquidity Cost}_i = \frac{1}{2} \times \text{spread}_i \times \text{stress multiplier}_i \times |MV_i|$$

LVaR is not a regulatory requirement. It originates from banking regulation
(Basel III internal models) and academic literature (Amihud & Mendelson, BIS
working papers) and is used internally by risk managers to capture the
liquidity dimension of market risk.

Spreads and stress multipliers are illustrative values adopted in this
notebook. In practice these are internal RMP parameters calibrated by the
fund manager and reviewed periodically.

| Asset Class      | Normal Spread | Stress Multiplier | Stressed Spread       |
|------------------|---------------|-------------------|-----------------------|
| Large cap equity | 5bps          | 3x                | 15bps                 |
| Small cap equity | 20bps         | 5x                | 100bps                |
| IG bonds         | 10bps         | 5x                | 50bps                 |
| HY bonds         | 50bps         | 10x               | 500bps                |
| Senior loans     | 100bps        | 20x               | 2000bps               |
| Listed REITs     | 15bps         | 5x                | 75bps                 |
| Direct property  | N/A           | N/A               | 5-8% transaction cost |

Mandatory liquidity risk management (buckets, redemption stress, LMTs) is
covered in Section 5.

In [ ]:
# MRS-33: Liquidity-adjusted VaR
lvar_result = liquidity_adjusted_var(var_1d, risk_df)

print(f"VaR (1d 99%)        : {lvar_result['var']*100:.2f}%   EUR {lvar_result['var']*NAV:,.0f}")
print(f"Liquidity cost      : {lvar_result['liquidity_cost']*100:.2f}%   EUR {lvar_result['liquidity_cost']*NAV:,.0f}")
print(f"LVaR (1d 99%)       : {lvar_result['lvar']*100:.2f}%   EUR {lvar_result['lvar']*NAV:,.0f}")
print(f"LVaR increase       : +{lvar_result['lvar_pct_increase']:.1f}%")
print()
print(lvar_result['by_asset_class'].to_string())

---

## 3. VaR Backtest 

VaR backtesting compares predicted VaR against realised daily P&L.
Two statistical tests are applied:

- **Kupiec POF**: tests whether the breach rate equals the expected rate (1% for 99% VaR)
- **Christoffersen**: tests whether breaches are independently distributed over time

> **Note**: Kupiec and Christoffersen tests are industry best practice and not explicitly
> required by ESMA or CSSF. The regulatory requirement is the 250-day breach count
> and zone classification only. Both tests should be documented in the RMP. For both
> models p > 0.05: fail to reject the null, model is statistically acceptable.

> **Private debt note**: NAV returns reflect mark-to-model valuations from the fund
> administrator, which smooth daily volatility. Breach counts will likely be lower
> than for a liquid fund. Results should be interpreted with this in mind.

ESMA (CESR/10-788) requires backtesting 1-day 99% VaR against realised daily P&L
over a 250 trading day rolling window.

Zone classification (Basel traffic light, adopted by ESMA):
- **Green** (0-4 breaches): model acceptable
- **Amber** (5-9 breaches): explanation required, possible model review
- **Red** (10+ breaches): model must be revised, CSSF notification required

#### 3.1. Internal (maximum history)

In [ ]:

# MRS-27: VaR backtest
nav_history = query_nav_history(ENGINE, FUND_ID)
nav_history['date'] = pd.to_datetime(nav_history['date'])

returns   = nav_history['pnl_pct'].dropna().values
dates = nav_history['date'].iloc[1:].reset_index(drop=True)

window        = 250
var_hist      = pd.Series([
    var_historical(returns[max(0, i-window):i], confidence=0.99)
    for i in range(window, len(returns))
])
var_param     = pd.Series([
    var_parametric(mu=0, sigma=returns[max(0, i-window):i].std(),
                   confidence=0.99, dist='t')
    for i in range(window, len(returns))
])

returns_aligned   = returns[window:]
dates_aligned = dates.iloc[window:].reset_index(drop=True)
print(f"Backtest observations : {len(returns_aligned)}")

In [ ]:
report = full_backtest_report(
    returns_series=pd.Series(returns_aligned),
    var_dict={'Historical': var_hist, 'Parametric': var_param},
    dates=dates_aligned
)
report[['n_obs', 'n_breaches', 'breach_rate', 'expected',
        'kupiec_p', 'christoffersen_p', 'result']]

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates_aligned, returns_aligned * 100,
        color=C['muted'], lw=0.8, label='Daily P&L', alpha=0.7)
ax.plot(dates_aligned, -var_hist * 100,
        color=ACCENT, lw=1.2, label='99% VaR (historical)')
ax.plot(dates_aligned, -var_param * 100,
        color=ACCENT3, lw=1.2, label='99% VaR (parametric)', linestyle='--')

breaches = returns_aligned < -var_hist.values
ax.scatter(dates_aligned[breaches], returns_aligned[breaches] * 100,
           color=ACCENT2, s=25, zorder=5, label=f'Breaches ({breaches.sum()})')

ax.set_ylabel('Daily P&L / VaR (%)', fontsize=9)
section_title(ax, f'VaR Backtest — {FUND_ID}')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
esma_250  = exception_report(pd.Series(returns_aligned[-250:]),
                              var_hist.iloc[-250:], confidence=0.99)
n_250     = len(esma_250)
breach_250 = n_250 / 250
zone_250  = 'Green' if n_250 <= 4 else 'Amber' if n_250 <= 9 else 'Red'

print(f"ESMA regulatory window — last 250 trading days")
print(f"Breaches    : {n_250}")
print(f"Breach rate : {breach_250*100:.2f}% (expected 1.0%)")
print(f"ESMA zone   : {zone_250}")
print()

#### 3.2. ESMA (250 trading days)

In [ ]:
dates_250 = dates_aligned.iloc[-250:].reset_index(drop=True)
returns_250   = returns_aligned[-250:]
var_250   = var_hist.iloc[-250:].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates_250, returns_250 * 100,
        color=C['muted'], lw=0.8, label='Daily P&L', alpha=0.7)
ax.plot(dates_250, -var_250 * 100,
        color=ACCENT, lw=1.2, label='99% VaR (historical)')

breaches_250 = returns_250 < -var_250.values
ax.scatter(dates_250[breaches_250], returns_250[breaches_250] * 100,
           color=ACCENT2, s=30, zorder=5, label=f'Breaches ({n_250})')

zone_color = {'Green': C['green'], 'Amber': C['amber'], 'Red': C['red']}
ax.set_title(f'ESMA Exception Report — Last 250 Days — Zone: {zone_250}',
             color=zone_color[zone_250], fontsize=11, pad=12)
ax.set_ylabel('Daily P&L / VaR (%)', fontsize=9)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---

## 4. Leverage (Annex IV)

AIFMD requires leverage to be reported using two methods:

- **Gross method**: sum of absolute exposures divided by NAV. No netting allowed.
  Derivatives converted to equivalent underlying exposure.
- **Commitment method**: hedging and netting arrangements are recognised.
  Offsetting positions in the same underlying reduce exposure.

Limits are set in the fund's offering document and reported quarterly to the CSSF
in Annex IV. AIFMD II (Directive 2024/927/EU) expanded reporting requirements,
including the breakdown by:
* asset class
* instrument type
* source: financial borrowing, synthetic leverage through derivatives, and repo/reverse repo.

The expanded disclosure makes it easier for regulators to identify funds building leverage through
derivatives rather than borrowing.

In [ ]:
# MRS-23: Leverage computation - Gross and Commitment method

# ----------------------------------------------------------------
# Gross method: sum of absolute exposures / NAV
# ----------------------------------------------------------------
risk_df['abs_exposure'] = risk_df['market_value_eur'].abs()

deriv_rows     = risk_df[risk_df['asset_class'] == 'Derivative'].copy()
deriv_notional = 0.0
for _, row in deriv_rows.iterrows():
    ticker        = 'SPXW 260619P05500 Index'
    bbg_data      = BBG.bdp(ticker, ['DELTA', 'OPT_UNDL_PX', 'CONTRACT_SIZE'])
    delta         = abs(bbg_data.loc[ticker, 'DELTA'])
    undl_px       = bbg_data.loc[ticker, 'OPT_UNDL_PX']
    contract_size = bbg_data.loc[ticker, 'CONTRACT_SIZE']
    quantity      = abs(row['quantity'])
    fx_rate       = 0.89
    deriv_notional += delta * quantity * contract_size * undl_px * fx_rate

risk_df['gross_exposure'] = risk_df.apply(
    lambda r: deriv_notional if r['asset_class'] == 'Derivative'
    else (0.0 if r['asset_class'] == 'Cash'
    else r['abs_exposure']),
    axis=1
)

gross_leverage = risk_df['gross_exposure'].sum() / NAV

# ----------------------------------------------------------------
# Commitment method
# ----------------------------------------------------------------
long_eq  = risk_df[(risk_df['asset_class'] == 'Equity') &
                   (risk_df['market_value_eur'] > 0)]['market_value_eur'].sum()
short_eq = risk_df[(risk_df['asset_class'] == 'Equity') &
                   (risk_df['market_value_eur'] < 0)]['market_value_eur'].sum()
net_eq   = abs(long_eq + short_eq)
bonds    = risk_df[risk_df['asset_class'].isin(['Bond','Loan','CLO'])]['market_value_eur'].abs().sum()
fx       = risk_df[risk_df['asset_class'] == 'FX']['market_value_eur'].abs().sum()

commitment_exposure = net_eq + bonds + fx + deriv_notional
commitment_leverage = commitment_exposure / NAV

# ----------------------------------------------------------------
# Summary table
# ----------------------------------------------------------------
all_classes = sorted(risk_df['asset_class'].unique())

leverage_summary = pd.DataFrame({
    'Gross (EUR)'        : [risk_df[risk_df['asset_class']==ac]['gross_exposure'].sum()
                            for ac in all_classes],
    'Gross (x NAV)'      : [risk_df[risk_df['asset_class']==ac]['gross_exposure'].sum()/NAV
                            for ac in all_classes],
    'Commitment (EUR)'   : [risk_df[risk_df['asset_class']==ac]['market_value_eur'].abs().sum()
                            if ac not in ['Cash', 'Derivative'] else
                            (deriv_notional if ac == 'Derivative' else 0)
                            for ac in all_classes],
    'Commitment (x NAV)' : [risk_df[risk_df['asset_class']==ac]['market_value_eur'].abs().sum()/NAV
                            if ac not in ['Cash', 'Derivative'] else
                            (deriv_notional/NAV if ac == 'Derivative' else 0)
                            for ac in all_classes],
}, index=all_classes)

leverage_summary['Gross (EUR)']        = leverage_summary['Gross (EUR)'].map('{:,.0f}'.format)
leverage_summary['Gross (x NAV)']      = leverage_summary['Gross (x NAV)'].map('{:.2f}x'.format)
leverage_summary['Commitment (EUR)']   = leverage_summary['Commitment (EUR)'].map('{:,.0f}'.format)
leverage_summary['Commitment (x NAV)'] = leverage_summary['Commitment (x NAV)'].map('{:.2f}x'.format)

print(f"{'Asset Class':<15} {'Gross (EUR)':>15} {'Gross':>8} {'Commit (EUR)':>15} {'Commit':>8}")
print('-' * 65)
for ac in all_classes:
    row = leverage_summary.loc[ac]
    print(f"{ac:<15} {row['Gross (EUR)']:>15} {row['Gross (x NAV)']:>8} "
          f"{row['Commitment (EUR)']:>15} {row['Commitment (x NAV)']:>8}")
print('-' * 65)
print(f"{'Total':<15} {risk_df['gross_exposure'].sum():>15,.0f} {gross_leverage:>7.2f}x "
      f"{commitment_exposure:>15,.0f} {commitment_leverage:>7.2f}x")

GROSS_LIMIT = 3.0
status      = 'OK' if gross_leverage <= GROSS_LIMIT else 'BREACH'
print(f"\nGross leverage limit : {GROSS_LIMIT:.0f}x")
print(f"Current gross        : {gross_leverage:.2f}x")
print(f"Status               : {status}")

In [ ]:
# AIFMD II granular leverage breakdown
granular = risk_df.groupby(['asset_class', 'sub_asset_class']).agg(
    gross_eur=('gross_exposure', 'sum'),
    n_positions=('isin', 'count')
).reset_index()
granular['gross_x_nav'] = granular['gross_eur'] / NAV
granular['source']      = granular.apply(
    lambda r: INSTRUMENT_SOURCE.get(
        (r['asset_class'], r['sub_asset_class']), ('Other', 'Other'))[0], axis=1)
granular['listed_otc']  = granular.apply(
    lambda r: INSTRUMENT_SOURCE.get(
        (r['asset_class'], r['sub_asset_class']), ('Other', 'Other'))[1], axis=1)

granular = granular[granular['gross_eur'] > 0].sort_values('gross_eur', ascending=False)

# listed vs OTC summary
total_gross = granular['gross_eur'].sum()
summary_lot = granular.groupby('listed_otc')['gross_eur'].sum().reset_index()
summary_lot['x_nav']        = summary_lot['gross_eur'] / NAV
summary_lot['pct_leverage'] = summary_lot['gross_eur'] / total_gross * 100
summary_lot['gross_eur']    = summary_lot['gross_eur'].map('{:,.0f}'.format)
summary_lot['x_nav']        = summary_lot['x_nav'].map('{:.2f}x'.format)
summary_lot['pct_leverage'] = summary_lot['pct_leverage'].map('{:.1f}%'.format)
summary_lot.index.name      = None
summary_lot.columns         = ['Category', 'Gross (EUR)', 'x NAV', '% Leverage']
summary_lot.set_index('Category', inplace=True)

header = f"{'':12} {'Gross (EUR)':>15} {'x NAV':>8} {'% Leverage':>12}"
print(header)
print('-' * len(header))
for idx, row in summary_lot.iterrows():
    print(f"{idx:<12} {row['Gross (EUR)']:>15} {row['x NAV']:>8} {row['% Leverage']:>12}")
print('-' * len(header))
print()

summary_src = granular.groupby('source')['gross_eur'].sum().reset_index()
summary_src['x_nav']        = summary_src['gross_eur'] / NAV
summary_src['pct_leverage'] = summary_src['gross_eur'] / total_gross * 100
summary_src['gross_eur']    = summary_src['gross_eur'].map('{:,.0f}'.format)
summary_src['x_nav']        = summary_src['x_nav'].map('{:.2f}x'.format)
summary_src['pct_leverage'] = summary_src['pct_leverage'].map('{:.1f}%'.format)
summary_src.set_index('source', inplace=True)
summary_src.index.name      = None

header = f"{'':20} {'Gross (EUR)':>15} {'x NAV':>8} {'% Leverage':>12}"
print(header)
print('-' * len(header))
for idx, row in summary_src.iterrows():
    print(f"{idx:<20} {row['gross_eur']:>15} {row['x_nav']:>8} {row['pct_leverage']:>12}")
print('-' * len(header))
print()

# granular table
granular['pct_leverage'] = (granular['gross_eur'] / total_gross * 100).map('{:.1f}%'.format)
granular['gross_eur']    = granular['gross_eur'].map('{:,.0f}'.format)
granular['gross_x_nav']  = granular['gross_x_nav'].map('{:.2f}x'.format)
granular.set_index(['source', 'asset_class', 'sub_asset_class'], inplace=True)
granular

---

## 4. Stress Testing (Annex VI)

AIFMD Annex VI requires AIFMs to conduct regular stress tests covering market,
liquidity, and counterparty risk. Scenarios must be documented in the RMP,
reviewed at least annually, and results reported to the CSSF via Annex IV.

Unlike UCITS, no specific scenarios are prescribed. The scenarios below reflect
market risk stresses appropriate for a private debt fund with senior loans,
HY bonds, and CLO exposure.

$$\Delta P_i = \text{sensitivity}_i \times \text{shock}_i \times MV_i$$

Scenarios covered:
- **Rate shock**: parallel shift up 200bps, impacting bond and CLO valuations
- **Credit widening**: spreads widen 150bps across loans, HY bonds and CLOs
- **Combined**: simultaneous rate and credit shock
- **Historical**: 2008 financial crisis, 2011 EU sovereign debt crisis, 2020 COVID crash, 2022 rate shock

> **Methodology note**: in this project, stress P&L uses first-order sensitivities
> (modified duration for rates and credit). Loans and CLOs are mark-to-model;
> stressed valuations may not reflect actual secondary market prices under stress.
> In production these figures come from a third-party risk system or modeled to higher orders.

In [ ]:
# MRS-26: Annex VI stress scenarios - private debt
from src.risk.risk_utils import HISTORICAL_SCENARIOS

# scenario table
rows = []
for key, p in HISTORICAL_SCENARIOS.items():
    rows.append({
        'Scenario'    : p['name'],
        'Equity'      : f"{p['delta_equity']*100:.0f}%",
        'Rates (bps)' : f"{p['delta_y']*10000:.0f}",
        'Credit (bps)': f"+{p['delta_spread']*10000:.0f}",
        'USD'         : f"{p['fx_shocks'].get('USD', 0)*100:+.0f}%",
        'GBP'         : f"{p['fx_shocks'].get('GBP', 0)*100:+.0f}%",
    })
pd.DataFrame(rows).set_index('Scenario')

In [ ]:
# stress results
rt  = stress_rates(risk_df, delta_y=0.02)
cr  = stress_credit(risk_df, delta_spread=0.015)
cb  = stress_combined(risk_df)
hist = {s: stress_historical(risk_df, s) for s in HISTORICAL_SCENARIOS}

rows = [
    {'Scenario': 'Rate Shock +200bps',      'P&L (EUR)': rt['stressed_pnl_eur'], '% NAV': rt['stressed_pnl_eur']/NAV*100},
    {'Scenario': 'Credit Widening +150bps', 'P&L (EUR)': cr['stressed_pnl_eur'], '% NAV': cr['stressed_pnl_eur']/NAV*100},
    {'Scenario': 'Combined',                'P&L (EUR)': cb['stressed_pnl_eur'], '% NAV': cb['stressed_pnl_eur']/NAV*100},
] + [
    {'Scenario': v['scenario'], 'P&L (EUR)': v['stressed_pnl_eur'], '% NAV': v['stressed_pnl_eur']/NAV*100}
    for v in hist.values()
]

summary_raw = pd.DataFrame(rows).set_index('Scenario')
worst_idx   = summary_raw['% NAV'].idxmin()

summary_raw['P&L (EUR)'] = summary_raw['P&L (EUR)'].map('{:,.0f}'.format)
summary_raw['% NAV']     = summary_raw['% NAV'].map('{:.2f}%'.format)

summary_raw.style.apply(lambda x: [
    'background-color: #7f1d1d; color: white' if i == worst_idx else ''
    for i in x.index], axis=0)

---

## 6. Liquidity Profile

ESMA guidelines (ESMA34-39-897) require AIFMs to categorise portfolio assets
by time to liquidation under normal market conditions. Results are reported
to the CSSF via Annex IV and used to assess redemption coverage.

Liquidity buckets (ESMA standard):

| Bucket | Time to liquidate |
|--------|------------------|
| 1      | 1 day            |
| 2      | 2-7 days         |
| 3      | 8-30 days        |
| 4      | 31-90 days       |
| 5      | 91-365 days      |
| 6      | > 1 year         |

ESMA buckets are roughly: day, week, month, quarter, year, or above.

Days to liquidate per position:

$$\text{days}_i = \frac{|MV_i|}{ADV_i \times \text{pct\_adv}}$$

* **ADV** (Average Daily Volume): average contracts/shares traded per day over
a 20-day window, sourced from Bloomberg. 
* **pct_adv = 25%** is an internal
RMP parameter representing the maximum fraction of ADV tradeable per day
without significant market impact. 
* Direct properties and instruments with
zero ADV are classified as illiquid (> 1 year).
* Cash and money market instruments are classified as 1 day by definition,
as they require no liquidation process.

In [ ]:

# MRS-30: Liquidity profile
risk_df_liq = days_to_liquidate(risk_df, pct_adv=0.25)
risk_df_liq = liquidity_buckets(risk_df_liq)

bucket_order = ['1 day', '2-7 days', '8-30 days', '31-90 days', '91-365 days', '> 1 year']

bucket_summary = risk_df_liq.groupby('liquidity_bucket').agg(
    market_value_eur=('market_value_eur', 'sum'),
    abs_exposure=('market_value_eur', lambda x: x.abs().sum()),
    n_positions=('isin', 'count')
).reset_index()
bucket_summary['pct_nav_net'] = bucket_summary['market_value_eur'] / NAV * 100
bucket_summary['pct_nav_abs'] = bucket_summary['abs_exposure'] / NAV * 100

bucket_full = bucket_summary.set_index('liquidity_bucket').reindex(bucket_order).fillna(0).reset_index()

# table
print(f"{'Bucket':<15} {'Abs Exposure (EUR)':>20} {'% NAV':>8} {'# Pos':>6}")
print('-' * 55)
for _, row in bucket_full.iterrows():
    if row['abs_exposure'] > 0:
        print(f"{row['liquidity_bucket']:<15} {row['abs_exposure']:>20,.0f} "
              f"{row['pct_nav_abs']:>7.1f}% {row['n_positions']:>6.0f}")
    else:
        print(f"{row['liquidity_bucket']:<15} {'—':>20} {'—':>8} {'—':>6}")
print('-' * 55)
total_abs = risk_df_liq['market_value_eur'].abs().sum()
print(f"{'Total':<15} {total_abs:>20,.0f} {total_abs/NAV*100:>7.1f}%")


# chart
fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(bucket_full['liquidity_bucket'],
              bucket_full['pct_nav_abs'],
              color=ACCENT, alpha=0.85, width=0.5)
ax.axhline(0, color=C['dim'], lw=0.8)
ax.set_ylabel('Absolute Exposure (% NAV)', fontsize=9)
section_title(ax, "Liquidity Profile — Absolute Exposure by Bucket")

for bar, val in zip(bars, bucket_full['pct_nav_abs']):
    if val > 2:
        ax.text(bar.get_x() + bar.get_width()/2, val/2,
                f'{val:.1f}%', ha='center', va='center',
                fontsize=8, color='white', fontweight='bold')
plt.tight_layout()
save_fig(fig, FUND_ID, "Liquidity buckets PE")
plt.show()

---
### 6.2 Redemption Stress — MRS-31

AIFMD Article 16 and ESMA/2020/1498 require AIFMs to assess whether the
fund can meet redemptions under normal and stressed conditions. Assets
liquidatable within the contractual notice period (buckets `'1 day'` and
`'2-7 days'`) are compared against the redemption amount. A shortfall
triggers a gate or side-pocket recommendation to the risk committee.

| Scenario | Redemption | Notice | Rationale |
| --- | --- | --- | --- |
| Normal | 10% NAV | 5 days | Routine investor exit |
| Large | 25% NAV | 5 days | Large single redemption |
| Stress | 50% NAV | 5 days | Co-ordinated stress exit |
| Largest investor | Fund-specific | 5 days | Concentration stress |

In [ ]:
# MRS-31: Redemption stress — AIFM Private Debt
_NOTICE = 5   # contractual notice period (days)
_SCENARIOS = [
    (0.10, 'Normal    (10% NAV)'),
    (0.25, 'Large     (25% NAV)'),
    (0.50, 'Stress    (50% NAV)'),
]

print(f'Fund: AIFM Private Debt  |  NAV: EUR {NAV:,.0f}  |  Notice: {_NOTICE} days')
print()
print(f"{'':22} {'Redemption (M)':>14} {'Liquid (M)':>12} {'Gap (M)':>12} {'Coverage':>10} Action")
print('\u2500' * 95)

_redstress = {}
for _pct, _label in _SCENARIOS:
    _r = redemption_stress(risk_df_liq, NAV, redemption_pct=_pct, notice_days=_NOTICE)
    _redstress[_pct] = _r
    _gap = f"+{_r['liquidity_gap_eur']/1e6:.1f}M" if _r['liquidity_gap_eur'] >= 0 else f"{_r['liquidity_gap_eur']/1e6:.1f}M"
    print(f"{_label:<22} {_r['redemption_amount_eur']/1e6:>13.1f}M {_r['liquid_assets_eur']/1e6:>11.1f}M "
          f"{_gap:>12} {_r['coverage_ratio']:>9.2f}x  {_r['recommendation']}")
print('\u2500' * 95)
print('Largest-investor stress: see Section 6.3')

### 6.3 Investor Concentration — MRS-32

**ESMA thresholds** (ESMA/2020/1498, Annex VI):
- Single investor **> 20% of NAV** → concentration risk flag
- Top 3 investors **> 50% of NAV** → high-concentration flag

A concentrated investor base amplifies redemption risk: one large exit
can force asset sales that affect all remaining investors. The largest-investor
scenario below connects MRS-31 and MRS-32 — it uses the investor register to
derive the fourth redemption stress scenario.

In [ ]:
# MRS-32: Investor concentration — AIFM Private Debt
_investors = pd.DataFrame([
    {'investor_id': 'PD001', 'investor_name': 'Nordic Sovereign Wealth Fund', 'investor_type': 'Sovereign Wealth', 'aum_eur': NAV * 0.35}, 
    {'investor_id': 'PD002', 'investor_name': 'European Insurance Alliance', 'investor_type': 'Insurance', 'aum_eur': NAV * 0.20}, 
    {'investor_id': 'PD003', 'investor_name': 'German Pension Consortium', 'investor_type': 'Pension Fund', 'aum_eur': NAV * 0.15}, 
    {'investor_id': 'PD004', 'investor_name': 'Danish Pension Fund A', 'investor_type': 'Pension Fund', 'aum_eur': NAV * 0.06}, 
    {'investor_id': 'PD005', 'investor_name': 'Institutional Investor B', 'investor_type': 'Asset Manager', 'aum_eur': NAV * 0.06}, 
    {'investor_id': 'PD006', 'investor_name': 'Insurance Holding C', 'investor_type': 'Insurance', 'aum_eur': NAV * 0.06}, 
    {'investor_id': 'PD007', 'investor_name': 'Family Office D', 'investor_type': 'Family Office', 'aum_eur': NAV * 0.06}, 
    {'investor_id': 'PD008', 'investor_name': 'Asset Manager E', 'investor_type': 'Asset Manager', 'aum_eur': NAV * 0.06}, 
])

_conc = investor_concentration(_investors, NAV, threshold=0.20)
_top  = _conc['by_investor']
_type = _investors.set_index('investor_id')['investor_type']

print(f'Investor Concentration — AIFM Private Debt  |  NAV: EUR {NAV:,.0f}')
print('ESMA threshold: 20% single / 50% top-3\n')
print(f"{'':4} {'Investor':<30} {'Type':<18} {'AUM (EUR)':>14} {'% NAV':>8}")
print('\u2500' * 80)
for _rank, (_, _row) in enumerate(_top.iterrows(), 1):
    _t = _type.get(_row['investor_id'], '')
    print(f"{_rank:<4} {_row['investor_name']:<30} {_t:<18} {_row['aum_eur']:>14,.0f} {_row['pct_nav']*100:>7.1f}%")
print('\u2500' * 80)

_flag_s = '\u26a0 ESMA flag'       if _conc['concentration_flag'] else '\u2713 OK'
_flag_3 = '\u26a0 High conc.'      if _conc['high_concentration'] else '\u2713 OK'
print(f"\nLargest investor : {_conc['largest_investor_pct']*100:.1f}% NAV  {_flag_s}")
print(f"Top 3 investors  : {_conc['top3_pct']*100:.1f}% NAV  {_flag_3}")

# Largest-investor redemption stress (4th scenario)
_r4   = redemption_stress(risk_df_liq, NAV, redemption_pct=_conc['largest_investor_pct'], notice_days=5)
_gap4 = f"+{_r4['liquidity_gap_eur']/1e6:.1f}M" if _r4['liquidity_gap_eur'] >= 0 else f"{_r4['liquidity_gap_eur']/1e6:.1f}M"
print(f"\nLargest-investor stress ({_conc['largest_investor_pct']*100:.1f}% NAV, 5-day notice):")
print(f"  Redemption : EUR {_r4['redemption_amount_eur']:,.0f}")
print(f"  Liquid     : EUR {_r4['liquid_assets_eur']:,.0f}")
print(f"  Gap        : {_gap4}  |  Coverage: {_r4['coverage_ratio']:.2f}x")
print(f"  Action     : {_r4['recommendation']}")

print('\nMonitoring recommendation:')
if _conc['high_concentration']:
    print('  \u2014 Enhanced monitoring: top-3 investors represent significant co-ordinated exit risk')
    print('  \u2014 Maintain liquidity buffer >= largest investor AUM')
if _conc['concentration_flag']:
    print(f'  \u2014 Gate-trigger review: largest investor at {_conc["largest_investor_pct"]*100:.1f}% NAV')
if not _conc['concentration_flag'] and not _conc['high_concentration']:
    print('  \u2014 No immediate action. Continue quarterly investor concentration monitoring.')

### 6.4 Counterparty Stress

For a private debt fund the primary counterparty risk is **borrower default**: the fund has extended loans to portfolio companies and the recovery in default depends on collateral quality and seniority. Senior secured loans typically achieve 40–60% recovery; mezzanine and second-lien positions recover less.

The stress scenario is the **largest single borrower defaulting** on its outstanding principal. Recovery is assumed at 40% (LGD 60%) consistent with ESMA guidance for senior secured private credit.


In [ ]:
import re

def extract_issuer(name):
    # remove CLO/ICS/fund suffixes and ratings/years
    name = re.sub(r'\b(CLO|ICS|AAA|AA\b|BBB|[A-Z]{1,3}\d+)\b.*', '', name)
    # remove coupon and maturity patterns like 6.5 2028
    name = re.sub(r'\s+\d+\.\d+\s+\d{4}.*', '', name)
    # remove standalone years
    name = re.sub(r'\s+\d{4}.*', '', name)
    return name.strip()

In [ ]:
# MRS-35: Counterparty stress — AIFM Private Debt
# Derive largest borrower exposure from live risk_df
_RECOVERY_RATE = 0.40  # senior secured LGD ~60%, ESMA benchmark

_loans_pd = risk_df[risk_df['market_value_eur'] > 0].copy()
_loans_pd ['issuer'] = _loans_pd ['instrument_name'].apply(extract_issuer)
_by_borrower = (
    _loans_pd
    .groupby('issuer', dropna=True)['market_value_eur']
    .sum()
    .reset_index()
    .rename(columns={'market_value_eur': 'exposure_eur'})
    .sort_values('exposure_eur', ascending=False)
    .reset_index(drop=True)
)
_by_borrower['loss_eur'] = _by_borrower['exposure_eur'] * (1 - _RECOVERY_RATE)
_by_borrower['exp_pct']  = _by_borrower['exposure_eur'] / NAV
_by_borrower['loss_pct'] = _by_borrower['loss_eur'] / NAV

_worst_borrow    = _by_borrower.iloc[0]
_borrow_loss_eur = _worst_borrow['loss_eur']
_borrow_loss_pct = _worst_borrow['loss_pct']

print(f"Counterparty Stress — AIFM Private Debt  |  NAV: EUR {NAV:,.0f}")
print(f"Senior secured recovery: {_RECOVERY_RATE*100:.0f}%  (LGD {(1-_RECOVERY_RATE)*100:.0f}%)\n")
print(f"{'Rank':<6} {'Issuer/Borrower':<28} {'Exposure (EUR)':>16} {'% NAV':>8} {'Loss (EUR)':>14} {'% NAV':>8}")
print('─' * 84)
for rank, (_, r) in enumerate(_by_borrower.head(5).iterrows(), 1):
    print(f"{rank:<6} {str(r['issuer']):<28} {r['exposure_eur']:>15,.0f} "
          f"{r['exp_pct']*100:>7.1f}%  {r['loss_eur']:>13,.0f} {r['loss_pct']*100:>7.1f}%")
print('─' * 84)
_limit_breach = _worst_borrow['exp_pct'] > 0.20
print(f"\nWorst-case: {_worst_borrow['issuer']} defaults")
print(f"  Exposure   : EUR {_worst_borrow['exposure_eur']:,.0f}  ({_worst_borrow['exp_pct']*100:.1f}% NAV)")
print(f"  Net loss   : EUR {_borrow_loss_eur:,.0f}  ({_borrow_loss_pct*100:.1f}% NAV)")
print(f"  Single-borrower limit (20% NAV): {"⚠ BREACH" if _limit_breach else "✓ Within limit"}")


### 6.5 Combined Stress (Market + Liquidity) — MRS-35

For private debt the relevant **market shock** is a credit spread widening (+150 bps), which reduces mark-to-market values of fixed-income loans. Combined with a **25% NAV redemption demand**, this is the most severe combined scenario. The private debt book is illiquid so stressed liquid assets are limited to cash and near-cash instruments only.


In [ ]:
# MRS-35: Combined stress — credit spread +150bps AND 25% redemption simultaneously
# cr is already in scope from Section 4 stress cell
_comb_mkt_eur_pd = cr['stressed_pnl_eur']
_comb_nav_st_pd  = NAV + _comb_mkt_eur_pd

# Under credit stress, liquid assets shrink proportionally with spread widening
_base_red_pd         = _redstress[0.25]
_comb_liquid_st_pd   = _base_red_pd['liquid_assets_eur'] * (1 - 0.10)  # 10% haircut on liquid book
_comb_redeem_eur_pd  = NAV * 0.25
_comb_gap_st_pd      = _comb_liquid_st_pd - _comb_redeem_eur_pd
_comb_cov_st_pd      = _comb_liquid_st_pd / _comb_redeem_eur_pd if _comb_redeem_eur_pd > 0 else float('inf')
_comb_action_pd      = 'Can meet redemption' if _comb_gap_st_pd >= 0 else 'Gate / partial suspension required'

print(f"Combined Stress — AIFM Private Debt  |  Credit +150bps + 25% Redemption")
print(f"Baseline NAV: EUR {NAV/1e6:,.1f}M\n")
print(f"  Market shock (credit spread +150bps):")
print(f"    Portfolio P&L  : EUR {_comb_mkt_eur_pd/1e6:,.1f}M  ({_comb_mkt_eur_pd/NAV*100:.1f}% NAV)")
print(f"    Stressed NAV   : EUR {_comb_nav_st_pd/1e6:,.1f}M")
print()
print(f"  Liquidity impact (25% redemption, liquid assets -10% haircut):")
print(f"    Redemption     : EUR {_comb_redeem_eur_pd/1e6:,.1f}M  (25% pre-stress NAV)")
print(f"    Liquid assets  : EUR {_comb_liquid_st_pd/1e6:,.1f}M  (post credit haircut)")
print(f"    Liquidity gap  : EUR {_comb_gap_st_pd/1e6:,.1f}M  |  Coverage: {_comb_cov_st_pd:.2f}x")
print(f"    Action         : {_comb_action_pd}")
print()
_total_stress_pd = _comb_mkt_eur_pd - max(0.0, -_comb_gap_st_pd)
_total_pct_pd    = _total_stress_pd / NAV * 100
print(f"  Total combined impact on NAV: EUR {_total_stress_pd/1e6:,.1f}M  ({_total_pct_pd:.1f}% of NAV)")
print(f"  Regulatory note: ESMA/2020/1498 §48 — combined stress is a mandatory Annex VI scenario")


## ESG Risk Indicators

ESG monitoring is required under CSSF Regulation 22-05 and SFDR (EU 2019/2088).
Portfolio-level ESG scores are computed as NAV-weighted averages. 

**SFDR classification**: Article 6 (no sustainability claim). ESG factors are
monitored but do not drive investment decisions. Article 8 would require documented
ESG screening; Article 9 would require sustainable investment as the primary objective.

Metrics monitored:
- Weighted average ESG score (composite, E, S, G)
- % of NAV in instruments with ESG score below 30*
- % of NAV with active controversy flag
- Weighted average carbon intensity (tCO2/EURm revenue)

> **Note**: ESG scores for liquid instruments are fetched from Bloomberg at
> enrichment time. Instruments without a Bloomberg ticker use fund administrator
> ESG data embedded in the position file.


> % of NAV in instruments with ESG score below internal threshold. 
> Threshold is defined in the Risk Management Policy. 
> ESG scores are not comparable across providers as each
> uses a different methodology and scale.
> (here using 30/100,Bloomberg scale 0-100 higher is better)

> **Scale note**: all ESG scores in this framework use a 0-100 scale where
> 100 is best, consistent with Bloomberg ESG methodology. Illiquid instrument
> scores are provided by the fund administrator on the same scale. In production,
> the ManCo should ensure all third-party ESG data is normalised to a consistent
> scale before aggregating portfolio-level metrics.

> **ESG look-through for derivatives**: derivatives have indirect ESG exposure
> through their underlying. The delta-adjusted notional is used as the ESG
> exposure weight rather than market value, which would understate the exposure.
>
> For options:
> $$ESG\_exposure_i = |\delta_i| \times Q_i \times \text{contract size} \times P_{underlying} \times FX$$
>
> For futures: full notional is used since delta = 1.
>
> For FX forwards: no ESG exposure assigned (no issuer).

In [ ]:
esg_df  = esg_u.build_esg_df(risk_df, BBG, ENGINE, FUND_ID, VALUATION_DATE)
esg_u.display_esg_assets(esg_df)

In [ ]:
esg_u.display_esg_summary(esg_df)

---

## 7. Annex IV Report — MRS-34

Private debt AIFs are subject to full Annex IV reporting under AIFMD Article 110.
The leverage section is particularly important: private debt funds can use both
financial borrowing (subscription credit lines, repo) and commitment-method leverage.
CSSF monitors whether actual leverage stays within the limits declared in the RMP.

For private debt, the commitment method recognises that loans are not easily netted,
so gross and commitment leverage are typically close to each other.

**Regulatory basis:** AIFMD Art. 110 / EU 231/2013 Annex IV / ESMA v1.7 (Jul 2024)


In [ ]:
from src.reporting.annex_iv import build_annex_iv, export_annex_iv_excel

ANNEX_IV_QUARTER = '2026-03-31'

annex_iv = build_annex_iv(ENGINE, FUND_ID, quarter=ANNEX_IV_QUARTER)
print(f"Annex IV — {FUND_ID}   Reporting period: Q1 2026 ({ANNEX_IV_QUARTER})")
print(f"NAV: EUR {annex_iv['_nav']/1e6:,.1f}M")

In [ ]:
def get_row_index(df, sections, colname):
    mask = df[colname].str.upper().isin([s.upper() for s in sections])
    section_rows = df[mask].index.tolist()
    return section_rows

sections_dict = {
    'identification':('field',['FUND IDENTITY', 'AIFM', 'COUNTERPARTIES', 
        'REPORTING', 'REDEMPTION TERMS', 'LEVERAGE LIMITS (RMP)']), 
    'breakdown':('Category',['Asset Class', 'Geography', 'Currency', 'Top 5 positions']),
    'risk_measures':('field', [f'VaR & ES (99% confidence, historical simulation, 250 days)', 'LEVERAGE', 'LIQUIDITY HEADLINE']
),
    'leverage_detail':('item',[
        f'GROSS METHOD — breakdown by source (EU231/2013 Art. 7)',
        f'COMMITMENT METHOD (EU231/2013 Art. 8)']),	
    'liquidity_buckets':None,
    'liquidity_terms':('field',['INVESTOR CONCENTRATION (ESMA thresholds)']),
    # '_nav':[],
 }

col_widths_bkdn = {
    'Category':'100px', 
    'NAV (EUR)':'200px', 
    'NAV %':'100px'
    }

def annex_iv_section(annex_iv, section, col_widths=None, fmt=None):
    
    df = annex_iv[section]
    df.columns = df.columns.str.replace('PCT', '%')
    col_align_override = {col:'right' for col in df.columns}
    
    section_rows = None
    if sections_dict[section]:
        colname, sub_sections = sections_dict[section]
        section_rows = get_row_index(df, sub_sections, colname)

    phtml.display_dark_table(
        df,
        caption=section.upper().replace('_',' '),
        fmt = fmt,
        # col_styles          : dict | None = None,
        col_align_override = col_align_override,
        highlight_rows = section_rows,
        col_widths=col_widths,
    )

In [ ]:
annex_iv_section(annex_iv, 'identification')

In [ ]:
print("\n=== Section 2.1 — Asset Class Breakdown ===")
display(annex_iv['asset_class_breakdown'])

print("\n=== Section 2.2 — Geographic Breakdown ===")
display(annex_iv['geography_breakdown'])

print("\n=== Section 2.4 — Top 5 Positions ===")
display(annex_iv['top5_positions'])


In [ ]:
print("\n=== Section 3 — Risk Measures ===")
display(annex_iv['risk_measures'])

print("\n=== Section 4 — Leverage Detail ===")
display(annex_iv['leverage_detail'])

print("\n=== Section 5 — Liquidity Profile ===")
liq = annex_iv['liquidity_buckets']
cols = [c for c in ['bucket', 'nav_eur', 'nav_pct', 'cumulative_pct'] if c in liq.columns]
display(liq[cols])

print("\n=== Section 5 — Redemption Terms & Investor Concentration ===")
display(annex_iv['liquidity_terms'])
